# Inspecting Data

You loaded a file. Now what? **Never clean or model data you have not inspected first.**
This chapter teaches how to understand any new dataset in about 5 minutes. In ML this step is called **EDA** (Exploratory Data Analysis) and it is the first thing interviewers expect you to do on a dataset.

**Contents**
1. The 5-minute inspection routine
2. Our dataset (messy on purpose)
3. First look: `head`, `tail`, `sample`, `shape`, `dtypes`
4. `info()`: how to read it
5. `describe()`: how to read it
6. Column inspection: `unique`, `nunique`, `value_counts`
7. Missing values
8. Duplicates
9. Statistics functions
10. Finding outliers
11. Separating numeric and categorical columns
12. Reusable `quick_report()` function
13. Display options
14. Pandas 3.x notes
15. Mini AI/ML example: audit before modelling
16. Common mistakes
17. Interview questions
18. Practice questions with solutions
19. Cheat sheet

In [63]:
import numpy as np
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

DATA = Path("data")
DATA.mkdir(exist_ok=True)

pandas: 3.0.6


---
# 1. The 5-minute inspection routine

Run these in order on **every** new dataset:

| Step | Question | Tool |
|---|---|---|
| 1 | How big is it? | `df.shape` |
| 2 | What does it look like? | `df.head()`, `df.sample()` |
| 3 | What are the columns and types? | `df.dtypes`, `df.info()` |
| 4 | What do the numbers look like? | `df.describe()` |
| 5 | What are the categories? | `value_counts()`, `nunique()` |
| 6 | What is missing? | `df.isna().sum()` |
| 7 | Any duplicate rows? | `df.duplicated().sum()` |
| 8 | Any outliers or impossible values? | `describe()`, `min/max`, IQR |
| 9 | How are columns related? | `df.corr()` |

Memory trick: **Shape -> Look -> Types -> Stats -> Categories -> Missing -> Duplicates -> Outliers -> Correlation**

---
# 2. Our dataset (messy on purpose)

We create a student performance dataset of 300 students and **deliberately put real-world problems in it**: missing values, duplicate rows, spelling mistakes in city names, and an impossible outlier.
Your job in this chapter is to **find all these problems using only inspection tools**.

(The next cell only creates the file. You can skip reading it.)

In [64]:
rng = np.random.default_rng(42)
n = 300

hours = rng.normal(5, 1.8, n).clip(0.5, 10).round(1)
attendance = rng.normal(78, 12, n).clip(35, 100).round(0)
prev_score = rng.normal(62, 14, n).clip(10, 100).round(0)
internet = rng.choice(["Yes", "No"], n, p=[0.8, 0.2])
score = (15 + 5.5 * hours + 0.25 * attendance + 0.2 * prev_score + rng.normal(0, 6, n)).clip(0, 100).round(0)

raw = pd.DataFrame({
    "student_id": np.arange(1001, 1001 + n),
    "age": rng.integers(18, 25, n),
    "city": rng.choice(["Indore", "Bhopal", "Delhi", "Pune"], n, p=[0.4, 0.25, 0.2, 0.15]),
    "hours_studied": hours,
    "attendance": attendance,
    "prev_score": prev_score,
    "internet": internet,
    "score": score,
})
raw["passed"] = np.where(raw["score"] >= 60, "Yes", "No")

# --- inject real-world problems ---
raw.loc[rng.choice(n, 25, replace=False), "attendance"] = np.nan        # missing
raw.loc[rng.choice(n, 14, replace=False), "prev_score"] = np.nan        # missing
raw.loc[[5, 17, 40], "city"] = ["indore", "Indore ", "BHOPAL"]          # inconsistent spelling
raw.loc[77, "hours_studied"] = 240.0                                    # impossible value
raw.loc[120, "age"] = -1                                                # impossible value
raw = pd.concat([raw, raw.sample(6, random_state=1)], ignore_index=True)  # 6 duplicate rows

raw.to_csv(DATA / "students_ml.csv", index=False)
print("file saved:", (DATA / "students_ml.csv").exists())

file saved: True


---
# 3. First look

In [65]:
df = pd.read_csv("data/students_ml.csv")
df.head()

,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
0,1001,19,Bhopal,5.5,99.0,69.0,Yes,76.0,Yes
1,1002,23,Bhopal,3.1,60.0,54.0,Yes,52.0,No
2,1003,22,Bhopal,6.4,88.0,80.0,Yes,97.0,Yes
3,1004,23,Indore,6.7,74.0,53.0,Yes,75.0,Yes
4,1005,18,Delhi,1.5,77.0,53.0,No,45.0,No


In [66]:
df.tail(3)

,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
303,1214,21,Indore,4.1,77.0,55.0,Yes,71.0,Yes
304,1107,19,Indore,3.3,77.0,49.0,Yes,66.0,Yes
305,1128,19,Indore,2.9,99.0,NaN,Yes,61.0,Yes


In [67]:
# head and tail show only the top/bottom, which can be misleading (data may be sorted).
# sample() gives random rows from everywhere
df.sample(5, random_state=7)

,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
125,1126,23,Bhopal,5.6,74.0,82.0,Yes,77.0,Yes
57,1058,23,Pune,7.7,81.0,64.0,Yes,92.0,Yes
285,1286,24,Pune,4.2,100.0,66.0,Yes,80.0,Yes
66,1067,21,Pune,4.4,69.0,39.0,No,69.0,Yes
114,1115,23,Bhopal,5.1,75.0,59.0,Yes,67.0,Yes


In [68]:
print("shape   :", df.shape)            # (rows, columns)
print("rows    :", len(df))
print("columns :", df.columns.tolist())
print("size    :", df.size)             # rows * columns
print("index   :", df.index)

shape   : (306, 9)
rows    : 306
columns : ['student_id', 'age', 'city', 'hours_studied', 'attendance', 'prev_score', 'internet', 'score', 'passed']
size    : 2754
index   : RangeIndex(start=0, stop=306, step=1)


In [69]:
df.dtypes

student_id         int64
age                int64
city                 str
hours_studied    float64
attendance       float64
prev_score       float64
internet             str
score            float64
passed               str
dtype: object

**What to notice here:** are the types what you expect? `student_id` is a number but it is just an identifier (we will never do maths on it). `passed` and `internet` are text categories. Everything else looks numeric.

---
# 4. `info()`: how to read it

In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   student_id     306 non-null    int64  
 1   age            306 non-null    int64  
 2   city           306 non-null    str    
 3   hours_studied  306 non-null    float64
 4   attendance     281 non-null    float64
 5   prev_score     291 non-null    float64
 6   internet       306 non-null    str    
 7   score          306 non-null    float64
 8   passed         306 non-null    str    
dtypes: float64(4), int64(2), str(3)
memory usage: 25.0 KB


How to read the output:

| Part | Meaning | What to check |
|---|---|---|
| `RangeIndex: 306 entries` | Total rows | Compare with the original source |
| `Non-Null Count` | Rows with a real value | If less than total, **column has missing values** |
| `Dtype` | Data type | Is a number column stored as text? Is a date stored as text? |
| `memory usage` | RAM used | Matters for big data |

Here: total is 306 rows, but `attendance` and `prev_score` have fewer non-null values, so they contain missing data.

In [71]:
# More accurate memory (counts text columns properly)
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   student_id     306 non-null    int64  
 1   age            306 non-null    int64  
 2   city           306 non-null    str    
 3   hours_studied  306 non-null    float64
 4   attendance     281 non-null    float64
 5   prev_score     291 non-null    float64
 6   internet       306 non-null    str    
 7   score          306 non-null    float64
 8   passed         306 non-null    str    
dtypes: float64(4), int64(2), str(3)
memory usage: 25.0 KB


In [72]:
# Memory used by each column
df.memory_usage(deep=True)

Index             132
student_id       2448
age              2448
city             4129
hours_studied    2448
attendance       2448
prev_score       2448
internet         3303
score            2448
passed           3332
dtype: int64

---
# 5. `describe()`: how to read it

In [73]:
df.describe()

,student_id,age,hours_studied,attendance,prev_score,score
count,306.000000,306.000000,306.000000,281.000000,291.000000,306.000000
mean,1150.650327,20.849673,5.703268,77.690391,60.683849,73.526144
std,86.078911,2.379616,13.542489,11.966630,14.010700,11.878661
min,1001.000000,-1.000000,0.500000,42.000000,25.000000,40.000000
25%,1077.250000,19.000000,3.800000,70.000000,51.000000,65.000000
50%,1150.500000,21.000000,4.850000,78.000000,61.000000,73.000000
75%,1223.750000,23.000000,5.900000,86.000000,70.000000,82.000000
max,1300.000000,24.000000,240.000000,100.000000,100.000000,100.000000


`describe()` gives for each **numeric** column:

| Row | Meaning |
|---|---|
| `count` | Non-missing values (compare with total rows) |
| `mean` | Average |
| `std` | Spread of the data |
| `min` / `max` | Smallest / biggest value |
| `25%`, `50%`, `75%` | Quartiles. **50% is the median** |

### The detective reading (this is the real skill)
- `age` -> **min is -1**. Age cannot be negative. Data error!
- `hours_studied` -> **max is 240**. Nobody studies 240 hours a day. Data error! Also see how far the max is from the 75% value.
- `attendance` -> count is less than 306, so **missing values**.
- **mean vs median (50%)**: if they are very different, the column is skewed or has outliers. Check `hours_studied`: the outlier pulls the mean up.
- `student_id` -> statistics of an ID are meaningless. Ignore.

In [74]:
# Include text columns as well
df.describe(include="all")

,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
count,306.000000,306.000000,306,306.000000,281.000000,291.000000,306,306.000000,306
unique,NaN,NaN,7,NaN,NaN,NaN,2,NaN,2
top,NaN,NaN,Indore,NaN,NaN,NaN,Yes,NaN,Yes
freq,NaN,NaN,111,NaN,NaN,NaN,243,NaN,272
mean,1150.650327,20.849673,NaN,5.703268,77.690391,60.683849,NaN,73.526144,NaN
std,86.078911,2.379616,NaN,13.542489,11.966630,14.010700,NaN,11.878661,NaN
min,1001.000000,-1.000000,NaN,0.500000,42.000000,25.000000,NaN,40.000000,NaN
25%,1077.250000,19.000000,NaN,3.800000,70.000000,51.000000,NaN,65.000000,NaN
50%,1150.500000,21.000000,NaN,4.850000,78.000000,61.000000,NaN,73.000000,NaN
75%,1223.750000,23.000000,NaN,5.900000,86.000000,70.000000,NaN,82.000000,NaN


In [75]:
# Only text columns (works for both old and new pandas)
df.describe(exclude="number")

,city,internet,passed
count,306,306,306
unique,7,2,2
top,Indore,Yes,Yes
freq,111,243,272


In [76]:
# Custom percentiles
df["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99])

count    306.000000
mean      73.526144
std       11.878661
min       40.000000
10%       59.000000
50%       73.000000
90%       89.000000
99%      100.000000
max      100.000000
Name: score, dtype: float64

---
# 6. Column inspection

## 6.1 `unique()` and `nunique()`

In [77]:
print(df["city"].unique())      # all different values
print(df["city"].nunique())     # how many different values

<ArrowStringArray>
['Bhopal', 'Indore', 'Delhi', 'indore', 'Pune', 'Indore ', 'BHOPAL']
Length: 7, dtype: str
7


**Problem found:** we expected 4 cities but see `indore`, `Indore ` (with a space) and `BHOPAL` as extra values. Same city written in different ways. A model would treat them as different cities.

## 6.2 `value_counts()` (your best EDA friend)

In [78]:
df["city"].value_counts()

city
Indore     111
Bhopal      85
Delhi       58
Pune        49
indore       1
Indore       1
BHOPAL       1
Name: count, dtype: int64

In [79]:
# As percentage
df["passed"].value_counts(normalize=True).round(3)

passed
Yes    0.889
No     0.111
Name: proportion, dtype: float64

In [80]:
# Also count missing values
df["attendance"].value_counts(dropna=False).head()

attendance
NaN     25
74.0    14
76.0    14
82.0    12
77.0    11
Name: count, dtype: int64

In [81]:
# value_counts on several columns together
df[["internet", "passed"]].value_counts()

internet  passed
Yes       Yes       216
No        Yes        56
Yes       No         27
No        No          7
Name: count, dtype: int64

In [82]:
# Check every text column at once
for col in df.select_dtypes(exclude="number").columns:
    print(f"--- {col} ---")
    print(df[col].value_counts().head(8))
    print()

--- city ---
city
Indore     111
Bhopal      85
Delhi       58
Pune        49
indore       1
Indore       1
BHOPAL       1
Name: count, dtype: int64

--- internet ---
internet
Yes    243
No      63
Name: count, dtype: int64

--- passed ---
passed
Yes    272
No      34
Name: count, dtype: int64



## 6.3 `nunique()` for the whole DataFrame

Useful to classify columns:
- `nunique` = number of rows -> probably an **ID** column
- `nunique` very small (2 to 10) -> **categorical** column
- `nunique` large with decimals -> **continuous numeric**
- `nunique` = 1 -> **useless** column (same value everywhere)

In [83]:
df.nunique()

student_id       300
age                8
city               7
hours_studied     75
attendance        50
prev_score        66
internet           2
score             53
passed             2
dtype: int64

---
# 7. Missing values

In [84]:
df.isna().sum()

student_id        0
age               0
city              0
hours_studied     0
attendance       25
prev_score       15
internet          0
score             0
passed            0
dtype: int64

In [85]:
# Percentage of missing per column (more useful than counts)
(df.isna().mean() * 100).round(2).sort_values(ascending=False)

attendance       8.17
prev_score       4.90
student_id       0.00
age              0.00
city             0.00
hours_studied    0.00
internet         0.00
score            0.00
passed           0.00
dtype: float64

In [86]:
# Total missing cells in the whole table
print(df.isna().sum().sum())

40


In [87]:
# Rows that have at least one missing value
rows_with_missing = df[df.isna().any(axis=1)]
print(len(rows_with_missing), "rows have at least one missing value")
rows_with_missing.head()

40 rows have at least one missing value


,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
7,1008,21,Delhi,4.4,NaN,68.0,Yes,76.0,Yes
13,1014,20,Delhi,7.0,83.0,NaN,Yes,90.0,Yes
20,1021,23,Indore,4.7,NaN,56.0,Yes,63.0,Yes
25,1026,20,Indore,4.4,NaN,55.0,Yes,80.0,Yes
26,1027,19,Pune,6.0,NaN,56.0,Yes,76.0,Yes


In [88]:
# Missing per row (how many gaps does each row have?)
df.isna().sum(axis=1).value_counts()

0    266
1     40
Name: count, dtype: int64

**Rule of thumb (for later chapters):**
- Missing below 5%: usually fill (mean/median) or drop rows
- Missing 5% to 40%: fill carefully, think about why it is missing
- Missing above 40-50%: consider dropping the column
- Fill/drop techniques are covered in chapter 07.

---
# 8. Duplicates

In [89]:
print("duplicate rows:", df.duplicated().sum())

duplicate rows: 6


In [90]:
# See the duplicate rows (keep=False shows ALL copies, including the first one)
df[df.duplicated(keep=False)].sort_values("student_id")

,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
106,1107,19,Indore,3.3,77.0,49.0,Yes,66.0,Yes
304,1107,19,Indore,3.3,77.0,49.0,Yes,66.0,Yes
123,1124,19,Indore,7.4,60.0,41.0,Yes,86.0,Yes
301,1124,19,Indore,7.4,60.0,41.0,Yes,86.0,Yes
127,1128,19,Indore,2.9,99.0,NaN,Yes,61.0,Yes
305,1128,19,Indore,2.9,99.0,NaN,Yes,61.0,Yes
185,1186,19,Bhopal,7.8,76.0,65.0,Yes,73.0,Yes
302,1186,19,Bhopal,7.8,76.0,65.0,Yes,73.0,Yes
189,1190,21,Delhi,8.0,86.0,60.0,Yes,93.0,Yes
300,1190,21,Delhi,8.0,86.0,60.0,Yes,93.0,Yes


In [91]:
# Duplicates based on selected columns only
# student_id must be unique, so repeated ids are suspicious
print(df.duplicated(subset=["student_id"]).sum())
print(df["student_id"].is_unique)

6
False


- `duplicated()` marks the **second and later** copies as `True` (first copy is `False`).
- `duplicated(keep=False)` marks **all** copies.
- Removing them: `df.drop_duplicates()` (covered in chapter 08).
- Duplicates are dangerous in ML: the same row can land in both train and test set and fake a good accuracy (data leakage).

---
# 9. Statistics functions

Each works on a Series (one column) or a DataFrame (all numeric columns).

In [92]:
s = df["score"]

print("count   :", s.count())          # non-null count
print("sum     :", s.sum())
print("mean    :", round(s.mean(), 2))
print("median  :", s.median())
print("mode    :", s.mode().tolist())  # most frequent value(s)
print("std     :", round(s.std(), 2))
print("var     :", round(s.var(), 2))
print("min/max :", s.min(), s.max())
print("range   :", s.max() - s.min())
print("q25/q75 :", s.quantile(0.25), s.quantile(0.75))
print("skew    :", round(s.skew(), 3))

count   : 306
sum     : 22499.0
mean    : 73.53
median  : 73.0
mode    : [64.0, 73.0]
std     : 11.88
var     : 141.1
min/max : 40.0 100.0
range   : 60.0
q25/q75 : 65.0 82.0
skew    : 0.003


In [93]:
# Position of the max/min
print("best student row :", df["score"].idxmax())
print(df.loc[df["score"].idxmax(), ["student_id", "score"]])

best student row : 30
student_id    1031.0
score          100.0
Name: 30, dtype: float64


In [94]:
# Several statistics at once
df[["hours_studied", "attendance", "score"]].agg(["mean", "median", "std", "min", "max"]).round(2)

,hours_studied,attendance,score
mean,5.70,77.69,73.53
median,4.85,78.00,73.00
std,13.54,11.97,11.88
min,0.50,42.00,40.00
max,240.00,100.00,100.00


In [95]:
# Top and bottom rows by a column
print(df.nlargest(3, "hours_studied")[["student_id", "hours_studied"]])
print()
print(df.nsmallest(3, "age")[["student_id", "age"]])

     student_id  hours_studied
77         1078          240.0
139        1140           10.0
248        1249           10.0

     student_id  age
120        1121   -1
4          1005   18
29         1030   18


## Correlation

Correlation tells how two numeric columns move together: **+1** (move together), **0** (no relation), **-1** (move opposite). It is the quickest way to see which features may help predict the target.

In [96]:
df[["age", "hours_studied", "attendance", "prev_score", "score"]].corr().round(2)

,age,hours_studied,attendance,prev_score,score
age,1.00,-0.08,0.12,0.01,-0.06
hours_studied,-0.08,1.00,-0.11,0.02,0.09
attendance,0.12,-0.11,1.00,-0.14,0.27
prev_score,0.01,0.02,-0.14,1.00,0.12
score,-0.06,0.09,0.27,0.12,1.00


In [97]:
# Only the relation with the target
df[["hours_studied", "attendance", "prev_score", "score"]].corr()["score"].sort_values(ascending=False).round(3)

score            1.000
attendance       0.274
prev_score       0.121
hours_studied    0.095
Name: score, dtype: float64

**Careful:** `hours_studied` shows a very weak correlation (about 0.09) with `score`. That looks strange, because studying more should help. The culprit is the impossible value `240`. Let us prove it:

In [98]:
with_outlier = df["hours_studied"].corr(df["score"])

clean_part = df[df["hours_studied"] < 24]          # drop the impossible rows
without_outlier = clean_part["hours_studied"].corr(clean_part["score"])

print("correlation with the outlier   :", round(with_outlier, 3))
print("correlation without the outlier:", round(without_outlier, 3))

correlation with the outlier   : 0.095
correlation without the outlier: 0.784


**One single bad row** changed the correlation from about 0.78 to 0.09. Outliers can completely hide real patterns. This is why we inspect before trusting any number.

Also remember: **correlation is not causation.**

---
# 10. Finding outliers

An **outlier** is a value far away from the rest. It can be a **data error** (age = -1) or a **real extreme** (a genius student).

## Method 1: look at min, max and sorted values

In [99]:
df["hours_studied"].sort_values(ascending=False).head(5)

77     240.0
248     10.0
139     10.0
226      9.5
30       8.9
Name: hours_studied, dtype: float64

## Method 2: IQR rule (the standard method)

1. `Q1` = 25th percentile, `Q3` = 75th percentile
2. `IQR = Q3 - Q1`
3. Lower limit = `Q1 - 1.5 * IQR`, upper limit = `Q3 + 1.5 * IQR`
4. Anything outside the limits is an outlier

In [100]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < low) | (series > high)
    return mask, low, high

mask, low, high = iqr_outliers(df["hours_studied"])
print(f"limits: {low:.2f} to {high:.2f}")
print("outliers:", mask.sum())
df[mask][["student_id", "hours_studied", "score"]]

limits: 0.65 to 9.05
outliers: 5


,student_id,hours_studied,score
77,1078,240.0,73.0
139,1140,10.0,100.0
226,1227,9.5,100.0
248,1249,10.0,100.0
293,1294,0.5,46.0


In [101]:
# Run on every numeric column
for col in ["age", "hours_studied", "attendance", "prev_score", "score"]:
    m, lo, hi = iqr_outliers(df[col])
    print(f"{col:15s} outliers: {m.sum():3d}   limits: {lo:7.1f} to {hi:7.1f}")

age             outliers:   1   limits:    13.0 to    29.0
hours_studied   outliers:   5   limits:     0.6 to     9.1
attendance      outliers:   1   limits:    46.0 to   110.0
prev_score      outliers:   1   limits:    22.5 to    98.5
score           outliers:   0   limits:    39.5 to   107.5


Not every outlier is wrong. IQR also flags a few legit extremes (like `prev_score`). **Decide with domain knowledge:**
- Impossible value (age -1, 240 hours/day) -> error, fix or remove
- Rare but possible value -> usually keep, or handle based on the model

## Method 3: logical checks (business rules)

In [102]:
# Rules we know must be true
print("negative ages        :", (df["age"] < 0).sum())
print("hours > 24 in a day  :", (df["hours_studied"] > 24).sum())
print("attendance > 100     :", (df["attendance"] > 100).sum())
print("score outside 0-100  :", (~df["score"].between(0, 100)).sum())

negative ages        : 1
hours > 24 in a day  : 1
attendance > 100     : 0
score outside 0-100  : 0


---
# 11. Separating numeric and categorical columns

In [103]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

print("numeric    :", numeric_cols)
print("categorical:", categorical_cols)

numeric    : ['student_id', 'age', 'hours_studied', 'attendance', 'prev_score', 'score']
categorical: ['city', 'internet', 'passed']


This split is used in almost every ML project: numeric columns get **scaling**, categorical columns get **encoding**.

**Think before trusting dtypes:**
- `student_id` is numeric but is an **identifier**: remove it from features.
- `passed` is the **target** (what we want to predict).
- A column like `pincode` is numeric but should be categorical/text.

---
# 12. Reusable `quick_report()` function

Build your own inspection helper once, then use it on **every** dataset in future projects.

In [104]:
def quick_report(df):
    print("=" * 50)
    print(f"Shape            : {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Duplicate rows   : {df.duplicated().sum()}")
    print(f"Missing cells    : {df.isna().sum().sum()}")
    print(f"Memory           : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print("=" * 50)

    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.count(),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
        "example": df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })
    return report

quick_report(df)

Shape            : 306 rows x 9 columns
Duplicate rows   : 6
Missing cells    : 40
Memory           : 25.0 KB


,dtype,non_null,missing,missing_%,unique,example
student_id,int64,306,0,0.0,300,1001
age,int64,306,0,0.0,8,19
city,str,306,0,0.0,7,Bhopal
hours_studied,float64,306,0,0.0,75,5.5
attendance,float64,281,25,8.2,50,99.0
prev_score,float64,291,15,4.9,66,69.0
internet,str,306,0,0.0,2,Yes
score,float64,306,0,0.0,53,76.0
passed,str,306,0,0.0,2,Yes


---
# 13. Display options

By default pandas hides middle columns/rows of big tables. You can change that:

In [105]:
pd.set_option("display.max_columns", 50)      # show up to 50 columns
pd.set_option("display.max_rows", 20)         # show up to 20 rows
pd.set_option("display.float_format", "{:.2f}".format)   # 2 decimals for floats

df.describe()

,student_id,age,hours_studied,attendance,prev_score,score
count,306.00,306.00,306.00,281.00,291.00,306.00
mean,1150.65,20.85,5.70,77.69,60.68,73.53
std,86.08,2.38,13.54,11.97,14.01,11.88
min,1001.00,-1.00,0.50,42.00,25.00,40.00
25%,1077.25,19.00,3.80,70.00,51.00,65.00
50%,1150.50,21.00,4.85,78.00,61.00,73.00
75%,1223.75,23.00,5.90,86.00,70.00,82.00
max,1300.00,24.00,240.00,100.00,100.00,100.00


In [106]:
# Reset only the options changed above
pd.reset_option("display.max_columns")
pd.reset_option("display.max_rows")
pd.reset_option("display.float_format")

---
# 14. Pandas 3.x notes

- Text columns have the dedicated string dtype (shown as `str`). Older tutorials use `select_dtypes(include="object")`. To stay safe in all versions, use **`select_dtypes(exclude="number")`** for text columns (as we did above), or `describe(exclude="number")`.
- `df.info()` shows `str` instead of `object` for text columns.
- Operations like `df.sort_values()` always return a **new** DataFrame. We never change the original by accident.

---
# 15. Mini AI/ML example: audit before modelling

Goal: predict whether a student **passes** (`passed`). Before any model, answer these audit questions with pandas.

In [107]:
data = pd.read_csv("data/students_ml.csv")

# 1. Size and duplicates
print("rows:", len(data), "| duplicates:", data.duplicated().sum())

# 2. Is the target balanced? (class imbalance changes how we train)
print()
print(data["passed"].value_counts(normalize=True).round(3))

rows: 306 | duplicates: 6

passed
Yes    0.889
No     0.111
Name: proportion, dtype: float64


In [108]:
# 3. Which columns have missing values and how many percent?
miss = (data.isna().mean() * 100).round(1)
miss[miss > 0]

attendance    8.2
prev_score    4.9
dtype: float64

In [109]:
# 4. How do features differ between pass and fail?
data.groupby("passed")[["hours_studied", "attendance", "prev_score"]].mean().round(1)

,hours_studied,attendance,prev_score
passed,,,
No,2.7,72.7,57.5
Yes,6.1,78.4,61.1


In [110]:
# 5. Columns to remove before training
drop_cols = ["student_id"]      # identifier, no predictive meaning
# "score" must also be removed if "passed" is the target.
# Reason: passed was made from score, so keeping it = DATA LEAKAGE
# (the model would just read the answer from the score column)
features = [c for c in data.columns if c not in drop_cols + ["score", "passed"]]
print("features to use:", features)

features to use: ['age', 'city', 'hours_studied', 'attendance', 'prev_score', 'internet']


**Audit result (what we would write in a project report):**
1. 306 rows, 6 exact duplicates -> remove
2. Target is imbalanced -> consider stratified split / class weights
3. `attendance` and `prev_score` have missing values -> impute
4. `city` has inconsistent spellings -> clean text
5. `age = -1` and `hours_studied = 240` are impossible -> fix
6. `student_id` is useless, `score` leaks the target -> drop from features

This is exactly the kind of analysis that makes a project stand out in an interview.

---
# 16. Common mistakes

| Mistake | Fix |
|---|---|
| Trusting `head()` only | Also use `sample()`, `info()`, `describe()` |
| Reading `describe()` of ID columns | Ignore IDs, they carry no meaning |
| Ignoring `count` row in `describe()` | Compare count with total rows to detect missing |
| Judging missing data by `info()` alone | Use `isna().sum()` and percentages |
| Assuming `value_counts()` counts NaN | Add `dropna=False` |
| Blindly removing every outlier | First decide: error or real extreme? |
| Computing correlation with outliers present | Handle outliers first (one bad row can hide a real pattern) |
| Keeping a column built from the target as a feature | Data leakage: drop it |
| `df.corr()` on text columns error | `df.corr(numeric_only=True)` or select numeric columns |
| Trusting dtypes blindly | An ID can be int, a pincode can be int; think about meaning |

---
# 17. Interview questions

**Q1. What are the first things you do when you get a new dataset?**
Check `shape`, `head`, `info`, `describe`, missing values, duplicates, value counts of categorical columns, outliers, and correlation with the target.

**Q2. Difference between `info()` and `describe()`?**
`info()` shows structure: columns, dtypes, non-null counts, memory. `describe()` shows statistics (count, mean, std, min, quartiles, max) of numeric columns.

**Q3. How do you find missing values?**
`df.isna().sum()` for counts and `df.isna().mean() * 100` for percentages.

**Q4. How do you detect outliers?**
Look at min/max and sorted values, use the IQR rule (`Q1 - 1.5*IQR`, `Q3 + 1.5*IQR`) or z-score, and apply business rules like age cannot be negative.

**Q5. `value_counts()` use?**
Frequency of each unique value. With `normalize=True` it gives proportions, with `dropna=False` it includes NaN. Used to check categories and class balance.

**Q6. `duplicated()` vs `drop_duplicates()`?**
`duplicated()` returns a boolean mask (True for repeated rows), `drop_duplicates()` removes them.

**Q7. What is data leakage?**
When a feature contains information that would not be available at prediction time, or is derived from the target. The model looks great in testing but fails in real life.

**Q8. Mean far greater than median. What does it mean?**
The data is right-skewed or has high outliers pulling the mean up.

**Q9. How to split numeric and categorical columns?**
`select_dtypes(include="number")` and `select_dtypes(exclude="number")`.

**Q10. What does a correlation of 0.8 / -0.8 / 0 mean?**
Strong positive / strong negative / no linear relationship. Correlation does not mean causation.

---
# 18. Practice questions

Use the same file. Run this cell first to get a fresh DataFrame.

In [111]:
df = pd.read_csv("data/students_ml.csv")

**Q1.** How many rows and columns are there? What is the memory used (in KB, deep)?
**Q2.** Which columns have missing values, and what is the missing percentage of each (only columns with missing > 0)?
**Q3.** How many students come from each city *after* fixing the spelling mistakes? (Hint: `str.strip().str.title()`)
**Q4.** Find the row(s) where `age` is less than 18.
**Q5.** What percentage of students have internet access?
**Q6.** Which 3 students have the highest `prev_score`? Show `student_id` and `prev_score`.
**Q7.** Using the IQR rule, how many outliers are in `attendance`?
**Q8.** Which numeric column has the strongest correlation with `score` (ignore `score` itself and `student_id`)?
**Q9.** Create a Series with the missing count of each column, sorted from highest to lowest.
**Q10.** Print the average `score` for students with and without internet.

### Solutions

In [112]:
# Q1
print(df.shape)
print(round(df.memory_usage(deep=True).sum() / 1024, 1), "KB")

(306, 9)
25.0 KB


In [113]:
# Q2
miss = (df.isna().mean() * 100).round(2)
miss[miss > 0]

attendance    8.17
prev_score    4.90
dtype: float64

In [114]:
# Q3
df["city"].str.strip().str.title().value_counts()

city
Indore    113
Bhopal     86
Delhi      58
Pune       49
Name: count, dtype: int64

In [115]:
# Q4
df[df["age"] < 18]

,student_id,age,city,hours_studied,attendance,prev_score,internet,score,passed
120,1121,-1,Indore,3.2,79.0,61.0,Yes,63.0,Yes


In [116]:
# Q5
(df["internet"].value_counts(normalize=True)["Yes"] * 100).round(1)

np.float64(79.4)

In [117]:
# Q6
df.nlargest(3, "prev_score")[["student_id", "prev_score"]]

,student_id,prev_score
246,1247,100.0
31,1032,98.0
76,1077,95.0


In [118]:
# Q7
q1, q3 = df["attendance"].quantile([0.25, 0.75])
iqr = q3 - q1
mask = (df["attendance"] < q1 - 1.5 * iqr) | (df["attendance"] > q3 + 1.5 * iqr)
print(mask.sum())

1


In [119]:
# Q8
corr = df.corr(numeric_only=True)["score"].drop(["score", "student_id"])
print(corr.round(3))
print("strongest:", corr.abs().idxmax())

age             -0.064
hours_studied    0.095
attendance       0.274
prev_score       0.121
Name: score, dtype: float64
strongest: attendance


In [120]:
# Q9
df.isna().sum().sort_values(ascending=False)

attendance       25
prev_score       15
student_id        0
age               0
city              0
hours_studied     0
internet          0
score             0
passed            0
dtype: int64

In [121]:
# Q10
df.groupby("internet")["score"].mean().round(2)

internet
No     73.78
Yes    73.46
Name: score, dtype: float64

---
# 19. Cheat sheet

```python
# Size and look
df.shape   len(df)   df.columns   df.index   df.dtypes
df.head()  df.tail()  df.sample(5, random_state=1)

# Structure and stats
df.info()  df.info(memory_usage="deep")
df.describe()   df.describe(include="all")   df.describe(exclude="number")

# Categories
s.unique()  s.nunique()  df.nunique()
s.value_counts(normalize=True, dropna=False)

# Missing and duplicates
df.isna().sum()   (df.isna().mean() * 100).round(2)
df[df.isna().any(axis=1)]
df.duplicated().sum()   df[df.duplicated(keep=False)]   df.drop_duplicates()

# Statistics
s.mean() s.median() s.mode() s.std() s.var() s.min() s.max() s.quantile(0.9) s.skew()
s.idxmax()  df.nlargest(3, "col")  df.nsmallest(3, "col")
df.agg(["mean", "std"])    df.corr(numeric_only=True)

# Outliers (IQR)
q1, q3 = s.quantile([0.25, 0.75]);  iqr = q3 - q1
mask = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

# Column types
df.select_dtypes(include="number")    df.select_dtypes(exclude="number")

# Display
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.2f}".format)

# Reset only changed options
pd.reset_option("display.max_columns")
pd.reset_option("display.max_rows")
pd.reset_option("display.float_format")
```

**Next chapter:** `05_selecting_data`: `loc`, `iloc`, boolean filtering, `isin`, `between`, `query`. Now that you know what is inside your data, you learn to pick exactly the rows and columns you want.